# Training-free DiT compression: GCP experiment notebook

> **Google Colab:** use **Runtime > Change runtime type > GPU**, then run the cells in **Part 0** below first. They clone this repository's branch, install dependencies and test the `loop_compression/` code (commits `9f0e4f3`, `929469d`). **Part X** then runs the five-method comparison (original, removal, loop, damped, XSA reuse) with optional Clean-FID/TPIPS. Sections 1-9 are the original standalone pilot; run them only if you want that experiment too.

Run in a **Python kernel with a CUDA GPU**. This notebook writes its helper module and CPU tests into the current folder, so uploading **this notebook alone** is sufficient.

Outputs: reproducible baseline images; feature-change heatmaps; original/removal/loop/damped comparison; paired LPIPS, parameter count, actual serialized checkpoint size, GPU memory and sampling latency. Everything is frozen; no training is performed.

Defaults are a **small pilot**: 50 sampling steps, 8 calibration images, 8 evaluation images, 4 removed blocks, fixed damping alpha=0.5. These are preliminary experiments, not FID or a tuned final model. Increase calibration coverage and evaluation size after the pilot works.

Reference implementations: [official DiT](https://github.com/facebookresearch/DiT), [DiT sampler](https://github.com/facebookresearch/DiT/blob/main/sample.py), [LPIPS](https://github.com/richzhang/PerceptualSimilarity). Whole-block residual damping here is our explicitly defined experiment, not a claimed reproduction of another paper's damping rule.


# Part 0: Colab setup and test of the looped-block compression commits

Tests the last two commits of branch `feature/looped-block-compression`:

| Commit | Content |
| --- | --- |
| `9f0e4f3` | `loop_compression/` (blocks, sensitivity, drive, final), FID/CLIP in `team_pipeline/metrics.py`, `--limit` and `fid` split support, tests |
| `929469d` | `.gitignore` only (nothing to run) |

Three levels, cheapest first: **A** CPU unit tests, **B** real DiT-XL/2 checks on the GPU, **C** end-to-end smoke runs through the shared runner (LPIPS, CLIP, FID, sensitivity, driver). Run every cell in this part in one session without restarting after Step 0.3, because the runner refuses to continue if the environment changed after `prepare`.

## 0.1 Settings and repository checkout

Edit the flags, then run. The cell is safe to re-run: it pulls new commits if the clone already exists. Push your latest local commits to GitHub first; Colab only sees what is on `origin`.

In [ ]:
import os, sys, subprocess, getpass, time
from pathlib import Path

# ---- settings -------------------------------------------------------------
REPO_URL = "https://github.com/cl-xy/cs5787-deep-learning.git"
BRANCH = "milestone-check"
PRIVATE_REPO = False      # True: prompts for a GitHub personal access token (not stored)
USE_DRIVE = True          # cache the 2.7 GB DiT checkpoint in Google Drive between sessions
# ---------------------------------------------------------------------------

IN_COLAB = "google.colab" in sys.modules


def sh(cmd, env=None, cwd=None, check=True):
    """Run a command and stream its output into the cell."""
    shell = isinstance(cmd, str)
    proc = subprocess.Popen(cmd, shell=shell, cwd=cwd, env={**os.environ, **(env or {})},
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in proc.stdout:
        print(line, end="")
    code = proc.wait()
    if check and code:
        raise subprocess.CalledProcessError(code, cmd)
    return code


if IN_COLAB:
    url = REPO_URL
    if PRIVATE_REPO:
        token = getpass.getpass("GitHub token: ")
        url = REPO_URL.replace("https://", f"https://{token}@")
    repo_dir = Path("/content/cs5787-deep-learning")
    if not repo_dir.exists():
        sh(["git", "clone", "--recurse-submodules", "-b", BRANCH, url, str(repo_dir)])
    else:
        sh(["git", "remote", "set-url", "origin", url], cwd=repo_dir)
        sh(["git", "fetch", "origin", BRANCH], cwd=repo_dir)
        sh(["git", "checkout", BRANCH], cwd=repo_dir)
        sh(["git", "pull", "--ff-only", "origin", BRANCH], cwd=repo_dir)
        sh(["git", "submodule", "update", "--init", "--recursive"], cwd=repo_dir)
    if PRIVATE_REPO:  # do not leave the token in .git/config
        sh(["git", "remote", "set-url", "origin", REPO_URL], cwd=repo_dir)
    os.chdir(repo_dir)
else:
    print("Not on Colab: assuming the working directory is the repository root.")
    sh(["git", "submodule", "update", "--init", "--recursive"], check=False)

REPO_ROOT = Path.cwd().resolve()
assert (REPO_ROOT / "loop_compression" / "blocks.py").exists(), \
    "loop_compression/ is missing: is the branch pushed and is this the repository root?"
assert (REPO_ROOT / "third_party/DiT/models.py").exists(), "DiT submodule is empty"
print("\nRepository root:", REPO_ROOT)
sh("git log -3 --stat --format='%n%h %s'")

## 0.2 GPU, Drive cache and dependencies

Installs the packages in `requirements-mac.txt` except PyTorch, torchvision and Jupyter, which Colab already provides (CUDA build). If pip reports that it replaced a package that was already imported (typically `numpy`, `pandas` or `huggingface-hub`), choose **Runtime > Restart session** and re-run from Step 0.1.

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > pick a GPU, then re-run from 0.1"
props = torch.cuda.get_device_properties(0)
print("GPU:", torch.cuda.get_device_name(0), f"({props.total_memory / 2**30:.1f} GiB)")
print("PyTorch:", torch.__version__, "| CUDA build:", torch.version.cuda)
if props.total_memory < 14 * 2**30:
    print("WARNING: under 14 GiB of GPU memory; DiT-XL/2 FP32 sampling plus the VAE may run out of memory.")

# Cache the checkpoint in Drive so a new Colab VM does not re-download 2.7 GB.
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    cache = Path("/content/drive/MyDrive/cs5787_cache/checkpoints")
    cache.mkdir(parents=True, exist_ok=True)
    link = REPO_ROOT / "checkpoints"
    if link.is_symlink() or not link.exists():
        if link.is_symlink():
            link.unlink()
        link.symlink_to(cache, target_is_directory=True)
    print("checkpoints/ ->", link.resolve())

# Dependencies: everything in requirements-mac.txt except torch, torchvision and Jupyter.
skip = ("torch=", "torch>", "torch<", "torchvision", "ipykernel", "jupyterlab")
lines = [l.strip() for l in (REPO_ROOT / "requirements-mac.txt").read_text().splitlines()]
reqs = [l for l in lines if l and not l.startswith("#") and not l.lower().startswith(skip)]
Path("requirements-colab.txt").write_text("\n".join(reqs) + "\n")
sh([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"])
sh([sys.executable, "-m", "pip", "check"], check=False)

## Level A: CPU unit tests

Tiny models and synthetic images, no downloads. This covers `tests_loop/` (blocks, windows, sensitivity, driver, final) and `tests_team/` (pipeline, FID/CLIP metrics, XSA reuse and fixed block selection). The untracked local folders `tests/` and `dit_looping/` are not part of the pushed commits and are not run here. A pass checks implementation contracts, not image quality.

In [ ]:
sh([sys.executable, "-m", "pytest", "--import-mode=importlib", "-q", "-ra", "tests_team", "tests_loop"],
   env={"DIT_TEST_REPO": str(REPO_ROOT / "third_party/DiT"), "MPLBACKEND": "Agg"})
print("\nLevel A passed.")

## Level B: `loop_compression` on the real DiT-XL/2 weights (single forward passes)

Downloads the pretrained checkpoint if it is not cached (about 2.7 GB). Checks on the actual model, on the GPU:

1. A looped window with every block kept and `K=1`, naive, reproduces the original output (the wrapper adds no change).
2. The smoke config (`loop_compression/configs/lc_smoke.json`: window 12-15, kept 13 and 14, K=2, damped, h=0.5) discards blocks 12 and 15, keeps the original weight tensors, has 28 block calls, and gives a finite output that differs from the original.
3. Parameter count equals the original minus the two discarded blocks.

In [ ]:
import json
from team_pipeline import experiment as exp
from loop_compression.method import build_model

exp.configure_dit(REPO_ROOT / "third_party/DiT")
checkpoint = exp.download_checkpoint(REPO_ROOT / "checkpoints/DiT-XL-2-256x256.pt")
device = torch.device("cuda")
model = exp.load_model(checkpoint).to(device)
base = model.blocks                               # original blocks (ScheduledBlocks)
n_params = lambda m: sum(p.numel() for p in m.parameters())
orig_params = n_params(model)
block_params = n_params(base.retained["12"])

g = torch.Generator().manual_seed(0)
x = torch.randn(1, 4, 32, 32, generator=g).repeat(2, 1, 1, 1).to(device)  # CFG copies, as in the sampler
t = torch.tensor([500, 500], device=device)
y = torch.tensor([207, 1000], device=device)      # class + null label, as the sampler passes it

def forward():
    with torch.inference_mode():
        return model.forward_with_cfg(x, t, y, 4.0).float().cpu()

ref = forward()

# 1. identity configuration
model.blocks = base
model = build_model(model, dict(window=[12, 15], kept=[12, 13, 14, 15], K=1, strategy="naive"))
ident = forward()
max_diff = (ident - ref).abs().max().item()
print(f"identity config: max |diff| = {max_diff:.2e}")
assert max_diff < 1e-5, "wrapper changes the output even when nothing is discarded or repeated"

# 2 + 3. smoke config
cfg = json.loads((REPO_ROOT / "loop_compression/configs/lc_smoke.json").read_text())
model.blocks = base
model = build_model(model, cfg)
meta = model.blocks.metadata()
print("removed:", meta["removed"], "| kept:", meta["kept"], "| block calls:", len(model.blocks),
      "| unique blocks:", len(meta["retained"]))
assert meta["removed"] == [12, 15] and len(model.blocks) == 28 and len(meta["retained"]) == 26
for i in (13, 14):
    assert model.blocks.retained[str(i)] is base.retained[str(i)], "kept block weights were replaced"
assert n_params(model) == orig_params - 2 * block_params, "unique parameter count is wrong"
out = forward()
rel = ((out[:, :4] - ref[:, :4]).norm() / ref[:, :4].norm()).item()
print(f"smoke config: relative eps change vs original = {rel:.4f}")
assert torch.isfinite(out).all() and rel > 0
print(f"parameters: {orig_params:,} -> {n_params(model):,}")

del model, base, ref, ident, out
exp._cleanup()
print("\nLevel B passed.")

## Level C: end-to-end smoke runs through the shared runner

Switches below control the slower stages. All use `--limit` or small sizes, so they check that the pipeline works, not that the method is good. Expect roughly 20-40 s per generated image on a T4; the totals are noted per stage.

In [ ]:
RUN_LPIPS = True          # prepare + validation runs (original, removal, looped) + LPIPS, ~10 min
RUN_METRICS = True        # CLIP + FID-1K code path on 8 fid-split images x 2 variants, ~12 min
RUN_SENSITIVITY = True    # sensitivity blocks/windows + driver dry run; builds a 16-image trajectory cache, ~10 min
LIMIT_VALIDATION = 2      # images per validation run (the runner requires at least 2)
LIMIT_FID = 8             # images for the FID/CLIP smoke test

RUN = Path("team_runs") / ("colab_" + time.strftime("%Y%m%d_%H%M%S"))
CLI = [sys.executable, "-m", "team_pipeline.cli"]
LC = "loop_compression.method:build_model"
SMOKE_CFG = "loop_compression/configs/lc_smoke.json"

# Fresh process state: Level B left no GPU memory behind.
torch.cuda.empty_cache()
sh(CLI + ["prepare", "--output", str(RUN), "--device", "cuda:0"])
print("\nRun root:", RUN)

### C1. LPIPS comparison (`--limit` and `--method module:function` paths)

Runs the unmodified baseline, plain removal of blocks 12 and 15, and the looped smoke config (same two blocks discarded, same parameter count) on the first validation images, then compares LPIPS, parameters and FLOPs. The looped model should show the same unique parameters as removal and more FLOPs.

In [ ]:
import json
import pandas as pd
from IPython.display import display, Image

if RUN_LPIPS:
    lim = ["--split", "validation", "--limit", str(LIMIT_VALIDATION)]
    sh(CLI + ["run", "--output", str(RUN), *lim, "--method", "original", "--name", "original"])
    sh(CLI + ["run", "--output", str(RUN), *lim, "--method", "removal", "--removed", "12", "15",
              "--name", "removal_12_15"])
    sh(CLI + ["run", "--output", str(RUN), *lim, "--method", LC, "--method-config", SMOKE_CFG,
              "--name", "lc_smoke"])
    sh(CLI + ["evaluate", "--output", str(RUN), "--split", "validation",
              "--names", "original", "removal_12_15", "lc_smoke"])
    root = RUN / "validation"
    res = pd.read_csv(root / "results.csv").set_index("variant")
    cols = [c for c in ["lpips_mean", "unique_parameters", "dit_sampling_GFLOPs", "latency_mean_s"] if c in res]
    display(res[cols])
    assert res.loc["lc_smoke", "unique_parameters"] == res.loc["removal_12_15", "unique_parameters"]
    assert res.loc["lc_smoke", "unique_parameters"] < res.loc["original", "unique_parameters"]
    assert res.loc["lc_smoke", "dit_sampling_GFLOPs"] > res.loc["removal_12_15", "dit_sampling_GFLOPs"]
    assert (RUN / "validation/lc_smoke/manifest.json").exists() or any((RUN / "validation/lc_smoke").glob("*.json"))
    display(Image(filename=str(RUN / "validation/comparison_grid.png")))
    print("\nC1 passed.")

### C2. FID-1K and CLIP code path (`fid` split, `team_pipeline/metrics.py`)

Generates a few images on the `fid` split for the baseline and the looped config, then runs the CLIP score and the FID-vs-original computation. With so few images the FID value is meaningless (the covariance is rank deficient); the point is that both commands run, write `clip.json` / `fid.json`, and give finite numbers. The 2 GB ADM reference file is not needed here.

In [ ]:
if RUN_METRICS:
    lim = ["--split", "fid", "--limit", str(LIMIT_FID)]
    sh(CLI + ["run", "--output", str(RUN), *lim, "--method", "original", "--name", "original"])
    sh(CLI + ["run", "--output", str(RUN), *lim, "--method", LC, "--method-config", SMOKE_CFG,
              "--name", "lc_smoke"])
    sh(CLI + ["clip", "--output", str(RUN), "--split", "fid", "--names", "original", "lc_smoke",
              "--device", "cuda"])
    sh(CLI + ["fid", "--output", str(RUN), "--split", "fid", "--names", "original", "lc_smoke",
              "--device", "cuda"])
    for variant in ("original", "lc_smoke"):
        clip = json.loads((RUN / "fid" / variant / "clip.json").read_text())
        print(variant, "clip.json:", clip)
    fid = json.loads((RUN / "fid/lc_smoke/fid.json").read_text())
    print("lc_smoke fid.json:", fid)
    print("\nC2 passed.")

### C3. Sensitivity screening and the validation driver

`sensitivity blocks` and `windows` replay the original sampler on the calibration split once to cache its states (all 16 calibration images, the slow part), then score block and window candidates by one-step output error. `windows` is limited to 6 candidate configs here. `drive validate --dry-run` then lists the validation runs it would launch from the shortlist without generating anything. `drive freeze` and `final` need full validation results, so they are covered by the unit tests only.

In [ ]:
if RUN_SENSITIVITY:
    sh([sys.executable, "-m", "loop_compression.sensitivity", "blocks", "--output", str(RUN),
        "--images", "2", "--stride", "10"])
    sh([sys.executable, "-m", "loop_compression.sensitivity", "windows", "--output", str(RUN),
        "--images", "2", "--stride", "10", "--max-configs", "6"])
    sens = RUN / "profile/sensitivity"
    for name in ["block_sensitivity.csv", "window_sweep.csv", "shortlist.json", "block_sensitivity_heatmap.png"]:
        assert (sens / name).exists(), f"missing {name}"
    display(pd.read_csv(sens / "block_sensitivity.csv").round(4).head(28))
    display(Image(filename=str(sens / "block_sensitivity_heatmap.png")))
    sh([sys.executable, "-m", "loop_compression.drive", "validate", "--output", str(RUN),
        "--limit", str(LIMIT_VALIDATION), "--dry-run"])
    print("\nC3 passed.")

### Save results

Copies this smoke run's tables, manifests and previews (no raw tensors) to Google Drive so they survive the session.

In [ ]:
if IN_COLAB and USE_DRIVE:
    dest = Path("/content/drive/MyDrive/cs5787_cache/runs") / RUN.name
    dest.parent.mkdir(parents=True, exist_ok=True)
    sh(["rsync", "-a", "--exclude", "*.pt", f"{RUN}/", f"{dest}/"])
    print("Saved to", dest)
else:
    print("Results are in", RUN.resolve())
print("\nPart 0 finished. Continue with Part X for the XSA comparison; sections 1-9 are the original standalone pilot (optional).")

# Part X: Five-method comparison with XSA reuse (team protocol)

Compares, on the shared `team_protocol.json` splits and the same frozen blocks:

| Name | Method | At each removed slot |
| --- | --- | --- |
| `original` | unmodified DiT-XL/2 | (none removed) |
| `member1_removal` | `removal` | skip the block |
| `member2_loop` | `loop` | rerun the nearest preceding retained block |
| `member3_damped` | `damped` | `h + alpha * (B(h, c) - h)` with the preceding block |
| `member4_xsa_reuse` | `xsa_reuse` | rerun the preceding block with exclusive self-attention (each head's own value direction is projected out before the output projection) |

Code: `team_pipeline/xsa.py` (XSA), `team_pipeline/methods.py` (`ScheduledBlocks`), `team_pipeline/experiment.py` (fixed selection), `team_pipeline/extended_metrics.py` (optional Clean-FID/TPIPS). Everything runs through `python -m team_pipeline.cli`; unit tests are in `tests_team/test_xsa.py` and run with Level A above.

**Prerequisites:** run **0.1** and **0.2** first (Level A recommended). Levels B and C are not required. Run this part in one session: `prepare` records the environment and later steps refuse to run if it changes. Each method generates 16 validation images; expect roughly 20-40 s per image on a T4.

In [ ]:
# XSA SETTINGS + PREPARE — downloads the checkpoint if needed; no image generation.
import json, time
from pathlib import Path
import pandas as pd
from IPython.display import display, Image

XSA_RESUME = None            # e.g. "team_runs/xsa_20261116_120000" to continue an existing run
FIXED_REMOVED = [4, 7]       # zero-based blocks shared by removal / loop / damped / XSA
CALIBRATE_BLOCKS = False     # True: choose blocks by calibration profiling instead (extra inference)
REMOVE_COUNT = 2             # used only when CALIBRATE_BLOCKS=True
ALPHA = 0.5                  # damped residual scale; fixed pilot value, tune on validation only
LIMIT = None                 # e.g. 2 for a quick smoke run (the runner needs at least 2 images)
XSA_METHODS = [              # (CLI method, run name, extra arguments)
    ("original", "original", []),
    ("removal", "member1_removal", []),
    ("loop", "member2_loop", []),
    ("damped", "member3_damped", ["--alpha", str(ALPHA)]),
    ("xsa_reuse", "member4_xsa_reuse", []),
]
XSA_NAMES = [name for _, name, _ in XSA_METHODS]
CLI = [sys.executable, "-m", "team_pipeline.cli"]

XSA_RUN = Path(XSA_RESUME) if XSA_RESUME else Path("team_runs") / ("xsa_" + time.strftime("%Y%m%d_%H%M%S"))
if not XSA_RESUME:
    torch.cuda.empty_cache()
    sh(CLI + ["prepare", "--output", str(XSA_RUN), "--device", "cuda:0"])
print("Run root:", XSA_RUN.resolve())

### X.1 Select blocks

Default: write `profile/selection.json` with the fixed blocks above (no inference). With `CALIBRATE_BLOCKS=True`, profile the calibration split and keep the `REMOVE_COUNT` lowest-change blocks instead. A selection is frozen for its run; use a new run to change it.

In [ ]:
# SELECT BLOCKS
selection_path = XSA_RUN / "profile/selection.json"
if selection_path.exists():
    print("Using the frozen selection of this run.")
elif CALIBRATE_BLOCKS:
    sh(CLI + ["profile", "--output", str(XSA_RUN), "--remove-count", str(REMOVE_COUNT)])
else:
    sh(CLI + ["select", "--output", str(XSA_RUN), "--removed", *map(str, FIXED_REMOVED)])
print("Selection:", json.loads(selection_path.read_text()))
heatmap = XSA_RUN / "profile/feature_change_heatmap.png"
if heatmap.exists():
    display(Image(filename=str(heatmap)))

### X.2 Validation runs (all five methods)

Each method loads its own model, generates the same validation images, saves tensors, PNGs and measurements, then releases the model. Finished variant folders are skipped, so the cell can be rerun after an interruption. A folder left by a crashed run is not overwritten: inspect it, move it aside, then rerun.

In [ ]:
# RUN VALIDATION — original, removal, loop, damped, XSA reuse
limit = ["--limit", str(LIMIT)] if LIMIT else []
for method, name, extra in XSA_METHODS:
    folder = XSA_RUN / "validation" / name
    if (folder / "grid.png").exists():
        print("Skipping finished run:", name)
        continue
    sh(CLI + ["run", "--output", str(XSA_RUN), "--split", "validation", *limit,
              "--method", method, "--name", name, *extra])
    display(Image(filename=str(folder / "grid.png")))

### X.3 Evaluation: paired LPIPS, resources and comparison grid

LPIPS runs on CPU and may download AlexNet weights on first use. It measures distance from the original outputs, not absolute generation quality. Timing includes synchronized diffusion, VAE decode and CPU transfer, and excludes loading, warmup, serialization and metrics. DiT parameter and checkpoint counts exclude the shared VAE.

In [ ]:
# RUN EVALUATION
sh(CLI + ["evaluate", "--output", str(XSA_RUN), "--split", "validation", "--names", *XSA_NAMES])
results = pd.read_csv(XSA_RUN / "validation/results.csv").set_index("variant")
cols = [c for c in ["lpips_mean", "lpips_std", "unique_parameters", "parameter_reduction_pct",
                    "dit_sampling_GFLOPs", "latency_mean_s", "latency_speedup_x"] if c in results]
display(results[cols].round(4))
display(Image(filename=str(XSA_RUN / "validation/comparison_grid.png")))

### X.4 Optional: Clean-FID and TPIPS

Disabled by default: with the defaults below nothing is installed or loaded, and the LPIPS results are copied to `results_extended.csv`. Set `FID_REFERENCE_DIR` to a folder of **real** images (with a description) to enable Clean-FID, and/or set `TPIPS_FACTORS` to enable TPIPS against the matched original images. Results go to `results_extended.csv`, `paired_metrics_extended.csv` and `evaluation_extended.json`; the base LPIPS files are untouched.

Optional packages are installed only after the base evaluation. Installing them can change package versions recorded by `prepare`; if a later `run` refuses the changed environment, prepare a new run. With 16 images, FID is a noisy pilot diagnostic (the 2048-d covariance is rank deficient). For the final FID-500/CLIP study on the `fid` split, use `loop_compression.final fid` (see `TEAM_WORKFLOW.md`).

In [ ]:
# OPTIONAL METRICS — leave the defaults for LPIPS only.
FID_REFERENCE_DIR = None          # e.g. "/content/imagenet_real_256"; real images only
FID_REFERENCE_DESCRIPTION = None  # dataset, split, class mixture, crop/resize
TPIPS_FACTORS = []                # e.g. ["overall", "shape", "texture"]
FID_DEVICE = "cpu"
TPIPS_DEVICE = "cuda:0"

extra_packages = (["clean-fid"] if FID_REFERENCE_DIR else []) + (["tpips"] if TPIPS_FACTORS else [])
if extra_packages:
    sh([sys.executable, "-m", "pip", "install", "-q", *extra_packages, "scipy<1.18"])
command = CLI + ["extended", "--output", str(XSA_RUN), "--split", "validation", "--names", *XSA_NAMES,
                 "--fid-device", FID_DEVICE, "--tpips-device", TPIPS_DEVICE]
if FID_REFERENCE_DIR:
    command += ["--fid-reference-dir", str(FID_REFERENCE_DIR),
                "--fid-reference-description", str(FID_REFERENCE_DESCRIPTION)]
if TPIPS_FACTORS:
    command += ["--tpips-factors", *TPIPS_FACTORS]
sh(command)
display(pd.read_csv(XSA_RUN / "validation/results_extended.csv"))

### X.5 Held-out test (disabled until choices are frozen)

Enable only after choosing blocks and `ALPHA` on validation. Runs all five methods on the **64 fresh test images** and evaluates them. Do not tune on these results.

In [ ]:
# RUN HELD-OUT TEST — disabled until choices are frozen.
RUN_HELD_OUT_TEST = False
if RUN_HELD_OUT_TEST:
    for method, name, extra in XSA_METHODS:
        if not (XSA_RUN / "test" / name / "grid.png").exists():
            sh(CLI + ["run", "--output", str(XSA_RUN), "--split", "test",
                      "--method", method, "--name", name, *extra])
    sh(CLI + ["evaluate", "--output", str(XSA_RUN), "--split", "test", "--names", *XSA_NAMES])
    display(pd.read_csv(XSA_RUN / "test/results.csv"))
    display(Image(filename=str(XSA_RUN / "test/comparison_grid.png")))
else:
    print("Skipped held-out testing; freeze validation choices before enabling.")

### X.6 Save results

Copies the run's tables, manifests and previews (no raw tensors or checkpoints) to Google Drive.

In [ ]:
# SAVE — copies everything except .pt tensors to Drive when available.
if IN_COLAB and USE_DRIVE:
    dest = Path("/content/drive/MyDrive/cs5787_cache/runs") / XSA_RUN.name
    dest.parent.mkdir(parents=True, exist_ok=True)
    sh(["rsync", "-a", "--exclude", "*.pt", f"{XSA_RUN}/", f"{dest}/"])
    print("Saved to", dest)
else:
    print("Results are in", XSA_RUN.resolve())

## 1. Write helper files and install dependencies

The next cell creates `dit_looping/`, `tests/test_experiment.py`, and `requirements-experiment.txt` beside this notebook. Use a new experiment folder if those filenames already contain your own work. Keep the CUDA-compatible PyTorch/torchvision pair supplied by GCP. If installing changes packages already imported into this kernel, **restart the kernel**, then continue from section 2.


In [ ]:
from pathlib import Path
import sys, subprocess, os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
FILES = {'dit_looping/__init__.py': '"""DiT layer-looping experiment helpers."""\n', 'dit_looping/experiment.py': '"""Training-free DiT experiments. Uses the official DiT checkout, without editing it."""\nfrom __future__ import annotations\n\nimport gc\nimport hashlib\nimport importlib.metadata\nimport json\nimport os\nfrom pathlib import Path\nimport platform\nimport subprocess\nimport sys\nimport time\n\nos.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")\nimport numpy as np\nimport torch\nfrom torch import nn\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    path.write_text(json.dumps(value, indent=2, allow_nan=False) + "\\n")\n\n\ndef configure_dit(repo):\n    repo = Path(repo).resolve()\n    if not (repo / "models.py").exists():\n        raise FileNotFoundError(f"Official DiT checkout missing: {repo}")\n    if str(repo) not in sys.path:\n        sys.path.insert(0, str(repo))\n    # Avoid silently using an unrelated notebook module called \'models\'.\n    import models\n    if Path(models.__file__).resolve() != repo / "models.py":\n        raise RuntimeError("An unrelated \'models\' module is loaded. Restart the kernel.")\n    torch.backends.cuda.matmul.allow_tf32 = False\n    torch.backends.cudnn.allow_tf32 = False\n    torch.backends.cudnn.benchmark = False\n    torch.use_deterministic_algorithms(True)\n    return subprocess.check_output(["git", "-C", str(repo), "rev-parse", "HEAD"], text=True).strip()\n\n\ndef environment():\n    packages = {}\n    for name in ["torch", "torchvision", "timm", "diffusers", "accelerate", "lpips", "numpy"]:\n        packages[name] = importlib.metadata.version(name)\n    return dict(python=sys.version, platform=platform.platform(), packages=packages,\n                cuda=torch.version.cuda,\n                gpu=torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)\n\n\nclass ScheduledBlocks(nn.Module):\n    """Register each retained block ONCE; execute it at one or more depth slots.\n\n    Official DiT.forward iterates over model.blocks. This container preserves that\n    interface, while its state_dict contains only retained modules.\n    """\n    def __init__(self, blocks, removed=(), mode="original", alpha=0.5):\n        super().__init__()\n        blocks = list(blocks)\n        self.depth = len(blocks)\n        self.mode = mode\n        self.alpha = float(alpha)\n        self.removed = sorted(set(int(i) for i in removed))\n        if mode not in {"original", "removal", "loop", "damped"}:\n            raise ValueError(f"Unknown mode: {mode}")\n        if not 0 <= self.alpha <= 1:\n            raise ValueError("Damping alpha must be in [0, 1].")\n        if any(i < 0 or i >= self.depth for i in self.removed):\n            raise ValueError("Removed index outside block range.")\n        if mode == "original" and self.removed:\n            raise ValueError("Original model cannot have removed blocks.")\n        if 0 in self.removed:\n            raise ValueError("Keep block 0: a replacement needs a preceding retained block.")\n        self.retained = nn.ModuleDict({str(i): b for i, b in enumerate(blocks)\n                                       if i not in self.removed})\n        self.schedule = []\n        previous = None\n        for i in range(self.depth):\n            if i not in self.removed:\n                previous = i\n                self.schedule.append((i, 1.0, False))\n            elif mode != "removal":\n                self.schedule.append((previous, self.alpha if mode == "damped" else 1.0, True))\n\n    def __len__(self):\n        return len(self.schedule)\n\n    @staticmethod\n    def _damped(block, scale):\n        def call(x, c):\n            if scale == 0:\n                return x\n            return x + scale * (block(x, c) - x)\n        return call\n\n    def __iter__(self):\n        for index, scale, replacement in self.schedule:\n            block = self.retained[str(index)]\n            yield self._damped(block, scale) if replacement and scale != 1 else block\n\n    def metadata(self):\n        return dict(mode=self.mode, removed=self.removed, alpha=self.alpha,\n                    depth=self.depth, retained=list(map(int, self.retained.keys())),\n                    execution_schedule=[dict(source=i, scale=a, replacement=r)\n                                        for i, a, r in self.schedule])\n\n\nclass FeatureProfiler:\n    """Per-sample L1 ratios, averaged separately over CFG\'s two branches.\n\n    Hooks run only on original-model calibration trajectories. Scalar tensors\n    stay on the GPU until collection ends, avoiding per-block CPU synchronization.\n    """\n    def __init__(self, model):\n        self.model = model\n        self.handles = []\n        self.records = []\n        self.timestep = None\n\n    def __enter__(self):\n        if self.model.blocks.mode != "original":\n            raise ValueError("Profile original model only.")\n        # Official forward_with_cfg calls self.forward directly, bypassing root\n        # Module hooks. The timestep embedder is still invoked through __call__.\n        self.handles.append(self.model.t_embedder.register_forward_pre_hook(self._before))\n        for index, block in self.model.blocks.retained.items():\n            def hook(module, args, output, index=int(index)):\n                x = args[0].float()\n                ratio = (output.float() - x).abs().flatten(1).sum(1) / (\n                    x.abs().flatten(1).sum(1) + 1e-8)\n                cond, uncond = ratio.chunk(2)\n                self.records.append(torch.stack([\n                    self.timestep.float(), ratio.new_tensor(index), cond.mean(), uncond.mean()\n                ]).detach())\n            self.handles.append(block.register_forward_hook(hook))\n        return self\n\n    def _before(self, module, args):\n        self.timestep = args[0][0].detach()\n\n    def __exit__(self, *exc):\n        for handle in self.handles:\n            handle.remove()\n        self.handles.clear()\n\n    def frame(self):\n        import pandas as pd\n        if not self.records:\n            raise RuntimeError("No profile records collected.")\n        values = torch.stack(self.records).cpu().numpy()\n        frame = pd.DataFrame(values, columns=["timestep", "block", "conditional", "unconditional"])\n        frame[["timestep", "block"]] = frame[["timestep", "block"]].astype(int)\n        frame["mean_cfg"] = frame[["conditional", "unconditional"]].mean(axis=1)\n        return frame\n\n\ndef summarize_profile(frame, output, remove_count=4):\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    import pandas as pd\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    # Average samples within each timestep first, then give each timestep equal weight.\n    matrix = frame.groupby(["block", "timestep"])["mean_cfg"].mean().unstack()\n    matrix = matrix.reindex(columns=sorted(matrix.columns, reverse=True))\n    if matrix.isna().any().any() or not np.isfinite(matrix.to_numpy()).all():\n        raise ValueError("Incomplete or nonfinite feature profile.")\n    scores = matrix.mean(axis=1).rename("mean_feature_change").to_frame()\n    depth = len(scores)\n    candidates = scores.loc[~scores.index.isin([0, depth - 1])]\n    if not 0 < remove_count <= len(candidates):\n        raise ValueError("Invalid removal budget.")\n    removed = sorted(candidates.sort_values("mean_feature_change", kind="stable").head(remove_count).index.tolist())\n    scores["selected"] = scores.index.isin(removed)\n    # Stages refer to sampling order, NOT increasing numerical diffusion t.\n    for stage, columns in zip(["early_noisy", "middle", "late_clean"],\n                              np.array_split(matrix.columns.to_numpy(), 3)):\n        scores[stage] = matrix.loc[:, columns].mean(axis=1)\n    scores.to_csv(output / "block_scores.csv", index_label="block")\n    matrix.to_csv(output / "feature_change_matrix.csv")\n    write_json(output / "selection.json", dict(removed=removed, protected=[0, depth - 1],\n        scoring="mean of per-sample normalized L1 changes, equally averaged over CFG branches and timesteps"))\n    fig, axes = plt.subplots(1, 2, figsize=(15, 7), constrained_layout=True)\n    for ax, values, title in zip(axes, [matrix.to_numpy(), np.log10(matrix.to_numpy().clip(1e-12))],\n                                  ["Mean normalized feature change", "log10 normalized feature change"]):\n        im = ax.imshow(values, aspect="auto", interpolation="nearest")\n        ticks = np.unique(np.linspace(0, len(matrix.columns) - 1, min(6, len(matrix.columns))).astype(int))\n        ax.set_xticks(ticks, [str(matrix.columns[i]) for i in ticks])\n        ax.set_yticks(range(depth))\n        ax.set_xlabel("Diffusion timestep (sampling proceeds left to right)")\n        ax.set_ylabel("Block index (zero based)")\n        ax.set_title(title)\n        for i in removed:\n            ax.axhline(i, color="red", linewidth=0.6, alpha=0.6)\n        fig.colorbar(im, ax=ax)\n    fig.savefig(output / "feature_change_heatmap.png", dpi=180)\n    plt.close(fig)\n    return removed, scores\n\n\ndef load_model(checkpoint, removed=(), mode="original", alpha=0.5):\n    from models import DiT_XL_2\n    model = DiT_XL_2(input_size=32, num_classes=1000)\n    state = torch.load(checkpoint, map_location="cpu", weights_only=True, mmap=True)\n    model.load_state_dict(state, strict=True, assign=True)\n    del state\n    model.blocks = ScheduledBlocks(model.blocks, removed, mode, alpha)\n    return model.eval().requires_grad_(False)\n\n\ndef load_compact_checkpoint(path):\n    """Rebuild the schedule before loading; plain official DiT cannot load this format."""\n    from models import DiT_XL_2\n    payload = torch.load(path, map_location="cpu", weights_only=True, mmap=True)\n    meta = payload["schedule"]\n    model = DiT_XL_2(input_size=32, num_classes=1000)\n    model.blocks = ScheduledBlocks(model.blocks, meta["removed"], meta["mode"], meta["alpha"])\n    model.load_state_dict(payload["state_dict"], strict=True, assign=True)\n    return model.eval().requires_grad_(False)\n\n\ndef checkpoint_stats(model, path, keep=False):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    # CPU serialization happens before the model goes to GPU; no hidden GPU copies.\n    torch.save(dict(format_version=1, architecture="DiT-XL/2-256",\n                    schedule=model.blocks.metadata(), state_dict=model.state_dict()), path)\n    result = dict(unique_parameters=sum(p.numel() for p in model.parameters()),\n                  checkpoint_bytes=path.stat().st_size,\n                  parameter_bytes=sum(p.numel() * p.element_size() for p in model.parameters()),\n                  unique_blocks=len(model.blocks.retained),\n                  block_evaluations_per_forward=len(model.blocks))\n    if not keep:\n        path.unlink()\n    return result\n\n\ndef download_checkpoint(path):\n    path = Path(path)\n    if not path.exists():\n        path.parent.mkdir(parents=True, exist_ok=True)\n        torch.hub.download_url_to_file(\n            "https://dl.fbaipublicfiles.com/DiT/models/DiT-XL-2-256x256.pt", str(path))\n    return path\n\n\n@torch.inference_mode()\ndef generate(model, vae, diffusion, label, seed, cfg_scale=4.0, progress=False):\n    device = next(model.parameters()).device\n    # Independent CPU generator makes initial latents independent of prior work.\n    z = torch.randn((1, 4, 32, 32), generator=torch.Generator().manual_seed(int(seed))).to(device)\n    z = torch.cat([z, z], dim=0)\n    labels = torch.tensor([int(label), 1000], device=device)\n    # Reset the complete ancestral-sampler noise stream for every paired sample.\n    with torch.random.fork_rng(devices=[device.index or 0]):\n        torch.manual_seed(int(seed) + 1_000_000)\n        samples = diffusion.p_sample_loop(model.forward_with_cfg, z.shape, noise=z,\n            clip_denoised=False, model_kwargs=dict(y=labels, cfg_scale=cfg_scale),\n            progress=progress, device=device)\n        image = vae.decode(samples[:1] / 0.18215).sample.clamp(-1, 1)\n    if not torch.isfinite(image).all():\n        raise RuntimeError(f"Nonfinite image: label={label}, seed={seed}")\n    return image.cpu()\n\n\n@torch.inference_mode()\ndef warmup(model, vae, cfg_scale):\n    device = next(model.parameters()).device\n    z = torch.zeros(2, 4, 32, 32, device=device)\n    model.forward_with_cfg(z, torch.full((2,), 999, device=device, dtype=torch.long),\n                           torch.tensor([0, 1000], device=device), cfg_scale)\n    vae.decode(z[:1] / 0.18215)\n    torch.cuda.synchronize()\n\n\ndef _prepare(config, mode, removed, alpha):\n    if not torch.cuda.is_available():\n        raise RuntimeError("Select a CUDA GPU kernel in GCP JupyterLab before running experiments.")\n    from diffusion import create_diffusion\n    from diffusers import AutoencoderKL\n    model = load_model(config["checkpoint"], removed, mode, alpha)\n    vae = AutoencoderKL.from_pretrained(config["vae_id"], revision=config.get("vae_revision"))\n    vae.eval().requires_grad_(False)\n    diffusion = create_diffusion(str(config["sampling_steps"]))\n    return model, vae, diffusion\n\n\ndef _cleanup():\n    gc.collect()\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n\n\ndef calibrate(config, pairs, output):\n    import pandas as pd\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    model, vae, diffusion = _prepare(config, "original", (), 1)\n    try:\n        model.cuda(); vae.cuda()\n        frames = []\n        for label, seed in pairs:\n            with FeatureProfiler(model) as profiler:\n                generate(model, vae, diffusion, label, seed, config["cfg_scale"])\n            frame = profiler.frame()\n            frame["label"], frame["seed"] = label, seed\n            frames.append(frame)\n            print(f"Profiled label={label}, seed={seed}", flush=True)\n            del profiler\n        frame = pd.concat(frames, ignore_index=True)\n        frame.to_csv(output / "feature_changes.csv", index=False)\n        write_json(output / "calibration_manifest.json", dict(config=config, pairs=pairs))\n        return frame\n    finally:\n        del model, vae, diffusion\n        _cleanup()\n\n\ndef run_variant(config, pairs, output, mode="original", removed=(), alpha=0.5,\n                keep_checkpoint=False):\n    import pandas as pd\n    from torchvision.utils import save_image\n    output = Path(output)\n    # Never silently mix outputs from different settings or overwrite prior results.\n    output.mkdir(parents=True, exist_ok=False)\n    model, vae, diffusion = _prepare(config, mode, removed, alpha)\n    try:\n        schedule = model.blocks.metadata()\n        stats = checkpoint_stats(model, output / "model.pt", keep_checkpoint)\n        write_json(output / "manifest.json", dict(config=config, pairs=pairs,\n                   schedule=schedule, stats=stats,\n                   vae_commit=getattr(vae.config, "_commit_hash", None),\n                   timing="synchronized diffusion + VAE decode + CPU image transfer; excludes loading, warmup, PNG/checkpoint writes, profiling and LPIPS"))\n        model.cuda(); vae.cuda()\n        warmup(model, vae, config["cfg_scale"])\n        rows, images = [], []\n        for index, (label, seed) in enumerate(pairs):\n            _cleanup()\n            torch.cuda.synchronize()\n            torch.cuda.reset_peak_memory_stats()\n            start = time.perf_counter()\n            image = generate(model, vae, diffusion, label, seed, config["cfg_scale"])\n            torch.cuda.synchronize()\n            elapsed = time.perf_counter() - start\n            row = dict(index=index, label=label, seed=seed, latency_s=elapsed,\n                       peak_allocated_bytes=torch.cuda.max_memory_allocated(),\n                       peak_reserved_bytes=torch.cuda.max_memory_reserved())\n            rows.append(row)\n            images.append(image)\n            torch.save(image, output / f"image_{index:04d}.pt")\n            save_image(image, output / f"image_{index:04d}.png", normalize=True, value_range=(-1, 1))\n            pd.DataFrame(rows).to_csv(output / "samples.csv", index=False)\n            print(f"{mode}: {index + 1}/{len(pairs)}, {elapsed:.2f} s", flush=True)\n        save_image(torch.cat(images), output / "grid.png", nrow=4,\n                   normalize=True, value_range=(-1, 1))\n        return pd.DataFrame(rows)\n    finally:\n        del model, vae, diffusion\n        _cleanup()\n\n\n@torch.inference_mode()\ndef compare_results(root, names):\n    """LPIPS runs on CPU after all GPU models are released; never contaminates timing."""\n    import lpips\n    import pandas as pd\n    root = Path(root)\n    metric = lpips.LPIPS(net="alex", version="0.1").cpu().eval()\n    baseline = pd.read_csv(root / "original" / "samples.csv")\n    ref_manifest = json.loads((root / "original" / "manifest.json").read_text())\n    all_rows, summary = [], []\n    for name in names:\n        folder = root / name\n        samples = pd.read_csv(folder / "samples.csv")\n        manifest = json.loads((folder / "manifest.json").read_text())\n        if manifest["config"] != ref_manifest["config"] or manifest["pairs"] != ref_manifest["pairs"]:\n            raise ValueError(f"Mismatched configuration or sample pairs: {name}")\n        if not samples[["index", "label", "seed"]].equals(baseline[["index", "label", "seed"]]):\n            raise ValueError(f"Mismatched sample order: {name}")\n        scores = []\n        for i in samples["index"]:\n            ref = torch.load(root / "original" / f"image_{i:04d}.pt", weights_only=True)\n            image = torch.load(folder / f"image_{i:04d}.pt", weights_only=True)\n            scores.append(float(metric(ref, image).item()))\n        samples["paired_lpips"] = scores\n        samples["variant"] = name\n        all_rows.append(samples)\n        stats = manifest["stats"]\n        summary.append(dict(variant=name, images=len(samples), lpips_mean=float(np.mean(scores)),\n            lpips_std=float(np.std(scores, ddof=1)) if len(scores) > 1 else 0.0,\n            unique_parameters=stats["unique_parameters"], unique_blocks=stats["unique_blocks"],\n            block_evaluations_per_forward=stats["block_evaluations_per_forward"],\n            checkpoint_MiB=stats["checkpoint_bytes"] / 2**20,\n            peak_GPU_allocated_GiB=samples["peak_allocated_bytes"].max() / 2**30,\n            peak_GPU_reserved_GiB=samples["peak_reserved_bytes"].max() / 2**30,\n            latency_mean_s=samples["latency_s"].mean(), latency_std_s=samples["latency_s"].std(ddof=0)))\n    result = pd.DataFrame(summary)\n    result.to_csv(root / "results.csv", index=False)\n    pd.concat(all_rows, ignore_index=True).to_csv(root / "paired_metrics.csv", index=False)\n    return result\n\n\ndef comparison_grid(root, names, pairs, count=8):\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    from PIL import Image\n    root = Path(root)\n    count = min(count, len(pairs))\n    fig, axes = plt.subplots(len(names), count, figsize=(2.4 * count, 2.7 * len(names)),\n                             squeeze=False, constrained_layout=True)\n    for row, name in enumerate(names):\n        for column in range(count):\n            ax = axes[row, column]\n            with Image.open(root / name / f"image_{column:04d}.png") as image:\n                ax.imshow(image)\n            ax.set_xticks([]); ax.set_yticks([])\n            if row == 0:\n                ax.set_title(f"class {pairs[column][0]}\\nseed {pairs[column][1]}", fontsize=9)\n            if column == 0:\n                ax.set_ylabel(name)\n    fig.savefig(root / "comparison_grid.png", dpi=160)\n    plt.close(fig)\n\n\ndef sha256(path):\n    digest = hashlib.sha256()\n    with open(path, "rb") as stream:\n        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n', 'requirements-experiment.txt': '# Keep the CUDA-enabled torch/torchvision pair supplied by your GCP image.\n# Install that pair first if missing: https://pytorch.org/get-started/locally/\ntorch>=2.2\ntorchvision>=0.17\ntimm==0.9.16\ndiffusers>=0.30,<0.36\naccelerate>=0.34,<2\nhuggingface-hub>=0.34,<1\nlpips==0.1.4\nnumpy>=1.26,<3\npandas>=2,<3\nmatplotlib>=3.8,<4\nPillow>=10\ntqdm>=4.66\npytest>=8,<10\n', 'tests/test_experiment.py': '"""CPU checks: no DiT weights, downloads, or CUDA needed."""\nimport json\nimport sys\nimport types\n\nimport numpy as np\nimport pytest\nimport torch\nfrom torch import nn\n\nfrom dit_looping.experiment import (\n    FeatureProfiler, ScheduledBlocks, checkpoint_stats, load_compact_checkpoint,\n    summarize_profile,\n)\n\n\nclass Affine(nn.Module):\n    def __init__(self, a, b):\n        super().__init__()\n        self.a = nn.Parameter(torch.tensor(float(a)))\n        self.b = nn.Parameter(torch.tensor(float(b)))\n\n    def forward(self, x, c):\n        return x * self.a + self.b\n\n\nclass TinyModel(nn.Module):\n    def __init__(self, **kwargs):\n        super().__init__()\n        self.t_embedder = nn.Identity()\n        self.blocks = nn.ModuleList([Affine(2, 1), Affine(3, 2), Affine(4, 3), Affine(5, 4)])\n\n    def forward(self, x, t, c):\n        self.t_embedder(t)\n        for block in self.blocks:\n            x = block(x, c)\n        return x\n\n\ndef scheduled(mode, alpha=0.5, removed=(1, 2)):\n    model = TinyModel()\n    model.blocks = ScheduledBlocks(model.blocks, removed, mode, alpha)\n    return model\n\n\ndef evaluate(model):\n    return model(torch.ones(2, 1), torch.tensor([999, 999]), None)\n\n\ndef test_recurrence_uses_updated_input_and_nearest_retained_block():\n    # Remove adjacent blocks 1 and 2: schedule 0,0,0,3, NOT 0,0,1,3.\n    model = scheduled("loop")\n    assert [i for i, _, _ in model.blocks.schedule] == [0, 0, 0, 3]\n    assert torch.equal(evaluate(model), torch.full((2, 1), 79.0))\n    assert len(model.blocks.retained) == 2\n    assert sum(p.numel() for p in model.parameters()) == 4\n    assert not any(key.startswith(("blocks.retained.1.", "blocks.retained.2."))\n                   for key in model.state_dict())\n\n\ndef test_damping_endpoints_and_intermediate_value():\n    assert torch.equal(evaluate(scheduled("damped", 0)), evaluate(scheduled("removal")))\n    assert torch.equal(evaluate(scheduled("damped", 1)), evaluate(scheduled("loop")))\n    # x=1 ->3 ->5 ->8 ->44 after two half-size repeated updates.\n    assert torch.equal(evaluate(scheduled("damped", 0.5)), torch.full((2, 1), 44.0))\n\n\ndef test_original_schedule_preserves_model():\n    model = TinyModel()\n    before = evaluate(model)\n    model.blocks = ScheduledBlocks(model.blocks)\n    assert torch.equal(before, evaluate(model))\n\n\ndef test_compact_checkpoint_roundtrip(tmp_path, monkeypatch):\n    monkeypatch.setitem(sys.modules, "models", types.SimpleNamespace(DiT_XL_2=TinyModel))\n    model = scheduled("damped")\n    compact = tmp_path / "compact.pt"\n    stats = checkpoint_stats(model, compact, keep=True)\n    restored = load_compact_checkpoint(compact)\n    assert torch.equal(evaluate(model), evaluate(restored))\n    assert restored.blocks.metadata() == model.blocks.metadata()\n    assert stats["checkpoint_bytes"] == compact.stat().st_size\n    assert stats["unique_parameters"] == 4\n    original = TinyModel()\n    original.blocks = ScheduledBlocks(original.blocks)\n    full_stats = checkpoint_stats(original, tmp_path / "full.pt", keep=True)\n    assert stats["checkpoint_bytes"] < full_stats["checkpoint_bytes"]\n\n\ndef test_profile_is_per_sample_and_separates_cfg_branches():\n    model = TinyModel()\n    model.blocks = ScheduledBlocks(model.blocks)\n    x = torch.tensor([[1.0], [2.0], [4.0], [8.0]])\n    with FeatureProfiler(model) as profiler:\n        model(x, torch.full((4,), 999), None)\n    frame = profiler.frame()\n    first = frame.iloc[0]\n    # First update is x+1, so normalized changes are 1 + 1/x.\n    assert first.conditional == pytest.approx((2 + 1.5) / 2)\n    assert first.unconditional == pytest.approx((1.25 + 1.125) / 2)\n    assert first.timestep == 999\n    assert len(frame) == 4\n    assert not model._forward_pre_hooks\n    assert not model.t_embedder._forward_pre_hooks\n    assert not model.blocks.retained["0"]._forward_hooks\n\n\ndef test_selection_protects_boundaries_and_orders_denoising_stages(tmp_path):\n    import pandas as pd\n    records = [dict(block=b, timestep=t, mean_cfg=(b + 1) * (t + 1))\n               for b in range(5) for t in [0, 499, 999]]\n    removed, scores = summarize_profile(pd.DataFrame(records), tmp_path, remove_count=2)\n    assert removed == [1, 2]\n    assert scores.loc[0, "early_noisy"] == 1000\n    assert scores.loc[0, "late_clean"] == 1\n    matrix = pd.read_csv(tmp_path / "feature_change_matrix.csv")\n    assert list(matrix.columns[1:]) == ["999", "499", "0"]\n    assert (tmp_path / "feature_change_heatmap.png").stat().st_size > 0\n    assert json.loads((tmp_path / "selection.json").read_text())["removed"] == [1, 2]\n\n\n@pytest.mark.parametrize("removed,mode,alpha", [([0], "loop", .5), ([4], "removal", .5),\n                                                   ([1], "original", .5), ([1], "damped", 1.5)])\ndef test_invalid_schedules_fail(removed, mode, alpha):\n    with pytest.raises(ValueError):\n        ScheduledBlocks(TinyModel().blocks, removed, mode, alpha)\n', 'tests/test_upstream_integration.py': '"""Small real DiT + sampler checks, no pretrained weights or GPU required.\n\nSet DIT_TEST_REPO to an official facebookresearch/DiT checkout to enable.\n"""\nimport copy\nimport os\nfrom pathlib import Path\n\nimport pytest\nimport torch\n\nfrom dit_looping.experiment import FeatureProfiler, ScheduledBlocks, configure_dit\n\n\ndef test_real_dit_and_sampler():\n    repo = os.environ.get("DIT_TEST_REPO")\n    if not repo:\n        pytest.skip("Set DIT_TEST_REPO to test against the official DiT source.")\n    configure_dit(Path(repo))\n    from models import DiT\n    from diffusion import create_diffusion\n    torch.set_num_threads(1)\n    torch.manual_seed(77)\n    original = DiT(input_size=8, patch_size=2, hidden_size=32,\n                   depth=6, num_heads=4).eval()\n    # Initialization is an identity with a zero output head. Make the test\n    # nontrivial without loading any large pretrained checkpoint.\n    with torch.no_grad():\n        for block in original.blocks:\n            torch.nn.init.normal_(block.adaLN_modulation[-1].weight, std=0.05)\n            torch.nn.init.normal_(block.adaLN_modulation[-1].bias, std=0.05)\n        torch.nn.init.normal_(original.final_layer.linear.weight, std=0.05)\n    wrapped = copy.deepcopy(original)\n    wrapped.blocks = ScheduledBlocks(wrapped.blocks)\n    x = torch.randn(2, 4, 8, 8)\n    t = torch.tensor([999, 999])\n    y = torch.tensor([207, 1000])\n    with torch.inference_mode():\n        ref = original.forward_with_cfg(x, t, y, 4.0)\n        assert torch.equal(ref, wrapped.forward_with_cfg(x, t, y, 4.0))\n        with FeatureProfiler(wrapped) as profiler:\n            wrapped.forward_with_cfg(x, t, y, 4.0)\n        profile = profiler.frame()\n        assert len(profile) == 6\n        assert profile["timestep"].eq(999).all()\n        outputs = {}\n        for mode, alpha in [("removal", 1), ("loop", 1), ("damped", 0), ("damped", 1)]:\n            model = copy.deepcopy(original)\n            model.blocks = ScheduledBlocks(model.blocks, [1, 2], mode, alpha)\n            outputs[mode, alpha] = model.forward_with_cfg(x, t, y, 4.0)\n        assert torch.equal(outputs["removal", 1], outputs["damped", 0])\n        assert torch.equal(outputs["loop", 1], outputs["damped", 1])\n        assert not torch.equal(ref, outputs["loop", 1])\n        # Original and scheduled-original follow the same actual sampler trajectory.\n        diffusion = create_diffusion("4")\n        trajectories = []\n        for model in [original, wrapped]:\n            torch.manual_seed(123)\n            trajectories.append(diffusion.p_sample_loop(\n                model.forward_with_cfg, x.shape, noise=x.clone(),\n                clip_denoised=False, model_kwargs=dict(y=y, cfg_scale=4.0),\n                progress=False, device="cpu"))\n        assert torch.equal(*trajectories)\n'}
for name, content in FILES.items():
    path = Path(name)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content)
subprocess.check_call([sys.executable, "-m", "pip", "install", "-r", "requirements-experiment.txt"])


## 2. Verify the GPU and fetch the official source

The official DiT repository and pretrained weights are downloaded from their original hosts. The repository commit, VAE revision, package versions, device, and checkpoint hash are recorded for reproducibility. Do not modify the upstream sampler or guidance implementation during this comparison.


In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import subprocess, sys, json
import shutil
import torch

print("Python kernel:", sys.executable)
print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print("CUDA_VISIBLE_DEVICES:", repr(os.environ.get("CUDA_VISIBLE_DEVICES")))
print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())
smi = shutil.which("nvidia-smi")
if smi is None and Path("/var/lib/nvidia/bin/nvidia-smi").exists():
    smi = "/var/lib/nvidia/bin/nvidia-smi"
if smi:
    try:
        report = subprocess.run([smi], capture_output=True, text=True, timeout=20)
        print(report.stdout or report.stderr)
    except subprocess.TimeoutExpired:
        print("nvidia-smi did not respond within 20 seconds.")
else:
    print("nvidia-smi was not found: check the VM GPU attachment and NVIDIA driver.")

if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU preflight failed. Inspect the diagnostics above. "
        "If nvidia-smi cannot list a GPU, check the GCP VM's GPU attachment/driver. "
        "If it lists a GPU but PyTorch CUDA build is None, this kernel has CPU-only PyTorch. "
        "Otherwise check the driver, CUDA build compatibility, and CUDA_VISIBLE_DEVICES. "
        "Changing a Jupyter kernel alone cannot add a GPU to a CPU-only VM."
    )
import torchvision
print("Torchvision:", torchvision.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))

REPO = Path("vendor/DiT").resolve()
if not REPO.exists():
    REPO.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["git", "clone", "https://github.com/facebookresearch/DiT.git", str(REPO)])

from dit_looping.experiment import (
    configure_dit, environment, write_json, download_checkpoint, sha256,
    run_variant, calibrate, summarize_profile, compare_results, comparison_grid,
    load_compact_checkpoint,
)
DIT_COMMIT = configure_dit(REPO)
print("DiT commit:", DIT_COMMIT)
# Tiny CPU tests require neither pretrained weights nor GPU inference.
subprocess.check_call([sys.executable, "-m", "pytest", "-q", "tests"],
                      env={**os.environ, "DIT_TEST_REPO": str(REPO), "MPLBACKEND": "Agg"})


## 3. Choose the experiment settings

Every evaluation image has its own label and seed. Initial noise and all ancestral-sampler noise are reset for each image, across all methods. Batch size is fixed at one image (two latent entries internally for classifier-free guidance). Sampling uses FP32, with TF32 disabled and deterministic algorithms enabled. Reproduction is intended for the recorded software/hardware environment; different GPUs or library versions may change numerics.

Keep calibration seeds separate from evaluation seeds. Alpha=0.5 is a fixed pilot setting; do not call it validation-selected. For a damping sweep, use a separate validation seed list, select alpha there, and evaluate the chosen value on fresh held-out seeds. Each run uses a new output folder to avoid mixing settings.


In [ ]:
from datetime import datetime, timezone
from huggingface_hub import model_info
import numpy as np

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
ROOT = Path("outputs") / RUN_ID
ROOT.mkdir(parents=True, exist_ok=False)

LABELS = [207, 360, 387, 974, 88, 979, 417, 279]  # ImageNet class indices
CALIBRATION_PAIRS = [(label, 10_000 + i) for i, label in enumerate(LABELS)]
EVALUATION_PAIRS = [(label, 30_000 + i) for i, label in enumerate(LABELS)]
REMOVE_COUNT = 4
ALPHA = 0.5
INCLUDE_RANDOM_CONTROL = False  # Optional fifth run: random selection + ordinary looping
KEEP_CHECKPOINTS = False  # True keeps several GB per variant; False measures then deletes

checkpoint = download_checkpoint(Path("checkpoints/DiT-XL-2-256x256.pt").resolve())
vae_id = "stabilityai/sd-vae-ft-mse"
CONFIG = dict(
    image_size=256, architecture="DiT-XL/2", checkpoint=str(checkpoint),
    checkpoint_sha256=sha256(checkpoint), dit_commit=DIT_COMMIT,
    sampling_steps=50, cfg_scale=4.0, sampler="official ancestral p_sample_loop",
    vae_id=vae_id, vae_revision=model_info(vae_id).sha,
    dtype="float32", tf32=False, deterministic_algorithms=True,
    batch_size_images=1, initial_noise_device="cpu", sampler_seed_offset=1_000_000,
)
assert not ({s for _, s in CALIBRATION_PAIRS} & {s for _, s in EVALUATION_PAIRS})
write_json(ROOT / "experiment.json", dict(
    config=CONFIG, environment=environment(), calibration_pairs=CALIBRATION_PAIRS,
    evaluation_pairs=EVALUATION_PAIRS, remove_count=REMOVE_COUNT, alpha=ALPHA,
    keep_checkpoints=KEEP_CHECKPOINTS, include_random_control=INCLUDE_RANDOM_CONTROL,
))
subprocess.run([sys.executable, "-m", "pip", "freeze"], stdout=(ROOT / "pip-freeze.txt").open("w"), check=True)
print("Output folder:", ROOT.resolve())


## 4. Reproducible original-model baseline

Model loading, checkpoint serialization, warmup, image writing and LPIPS are excluded from latency. Latency includes synchronized diffusion sampling, VAE decoding, and transfer of the decoded image to CPU. GPU memory includes the current DiT and VAE; no other variant stays loaded. Run without other workloads on the GPU. Per-image times are preliminary measurements across seeds, not a dedicated repeated benchmark.

Actual checkpoint bytes use the same serialization format for all variants. The VAE is shared and is **not** included in DiT parameter/checkpoint counts. Peak *allocated* memory counts live PyTorch tensors; *reserved* memory also includes its caching allocator. Other processes and driver memory are not captured by these numbers.


In [ ]:
baseline_samples = run_variant(CONFIG, EVALUATION_PAIRS, ROOT / "original",
                               keep_checkpoint=KEEP_CHECKPOINTS)
from IPython.display import display, Image
display(Image(filename=str(ROOT / "original/grid.png")))
display(baseline_samples)


## 5. Profile original-model calibration trajectories

For each sample, block and timestep, compute `sum(abs(h_out-h_in)) / (sum(abs(h_in)) + 1e-8)`. Average conditional and unconditional CFG branches equally; average samples within each timestep, then timesteps equally to rank blocks. Retain blocks 0 and 27. Red lines in the heatmap mark selected blocks.

Low change is a selection heuristic, not proof of unimportance. The heatmap shows update magnitude, not causal sensitivity. The downstream generation comparisons test whether the selected removals are actually tolerable. Early/noisy and late/clean stages follow sampling order, from high numerical timestep to low.


In [ ]:
profile = calibrate(CONFIG, CALIBRATION_PAIRS, ROOT / "profile")
REMOVED, block_scores = summarize_profile(profile, ROOT / "profile", REMOVE_COUNT)
print("Removed zero-based block indices:", REMOVED)
display(block_scores)
display(Image(filename=str(ROOT / "profile/feature_change_heatmap.png")))


## 6. Run the three compressed approaches

All three use exactly the same four removed blocks and 24 retained blocks. A missing slot reuses the nearest preceding **retained** block, including when multiple removed slots are adjacent. The original retained evaluations are unchanged. Each repeated evaluation receives the updated hidden state and the same timestep/class conditioning.

- Removal: skip missing slots (24 block evaluations per forward).
- Looping: execute a preceding retained block again (28 evaluations).
- Damped looping: at replacement slots only, apply `h + alpha * (B(h,c) - h)` (28 evaluations for alpha>0).

Retained weights are registered once and removed weights leave the model. Looping compresses unique weights; it does not promise lower FLOPs or latency. Damping at alpha=1 matches ordinary looping; alpha=0 matches removal.

A cell that fails partway leaves its outputs for inspection. To retry, rename that variant's partial folder, or start a fresh run from section 3. Existing variant folders are deliberately not overwritten.


In [ ]:
variants = [("removal", "removal", 1.0), ("loop", "loop", 1.0), ("damped", "damped", ALPHA)]
for name, mode, alpha in variants:
    run_variant(CONFIG, EVALUATION_PAIRS, ROOT / name, mode=mode,
                removed=REMOVED, alpha=alpha, keep_checkpoint=KEEP_CHECKPOINTS)


## 7. Paired metrics and comparison grid

LPIPS AlexNet v0.1 is computed on the saved FP32 decoded images in [-1,1], on CPU after GPU inference ends. Smaller LPIPS means closer agreement with the original output, not necessarily better generative quality. This small pilot does not establish distribution quality or class fidelity; inspect the image grid, and use a larger independent evaluation for final conclusions.


In [ ]:
NAMES = ["original", "removal", "loop", "damped"]
results = compare_results(ROOT, NAMES)
comparison_grid(ROOT, NAMES, EVALUATION_PAIRS)
display(results.round(4))
display(Image(filename=str(ROOT / "comparison_grid.png")))
print("Results CSV:", (ROOT / "results.csv").resolve())
print("Per-image metrics:", (ROOT / "paired_metrics.csv").resolve())


## 8. Optional random-selection control

Enable `INCLUDE_RANDOM_CONTROL` in section 3 before the run. This compares feature-guided ordinary looping against randomly selected blocks at the same weight budget. One draw is exploratory; repeat multiple selection seeds for a stronger control. Protected boundaries remain the same. The sampled selection is saved in the variant manifest.


In [ ]:
if INCLUDE_RANDOM_CONTROL:
    random_removed = sorted(np.random.default_rng(42).choice(
        np.arange(1, 27), size=REMOVE_COUNT, replace=False).tolist())
    run_variant(CONFIG, EVALUATION_PAIRS, ROOT / "random_loop", mode="loop",
                removed=random_removed, keep_checkpoint=KEEP_CHECKPOINTS)
    NAMES = ["original", "removal", "loop", "damped", "random_loop"]
    results = compare_results(ROOT, NAMES)
    comparison_grid(ROOT, NAMES, EVALUATION_PAIRS)
    display(results.round(4))
    display(Image(filename=str(ROOT / "comparison_grid.png")))


## 9. Reuse artifacts and plan the next experiment

Each variant folder contains a manifest, per-image seed/label/timing/memory CSV, individual PNGs and FP32 `.pt` images, plus its image grid. `profile/` contains raw feature-change records, the ranking, selected indices, stage summaries, and heatmaps. The run root contains environment and experiment settings, a package freeze, the results table, paired metrics, and the comparison grid.

If `KEEP_CHECKPOINTS=True`, load a compact model with the helper below. A compressed checkpoint needs its saved schedule; loading it directly into plain official DiT would be incorrect.

Before drawing stronger conclusions: expand calibration classes/seeds, compare budgets of 2/4/7 blocks, choose damping on disjoint validation samples, repeat random selection, and evaluate a larger held-out set. Change one factor at a time and use a new run folder for each experiment. For closer alignment with the official baseline sampling settings, repeat at 250 steps; do not mix 50-step and 250-step results in one comparison.

Use this notebook as implementation assistance. Write your own course report and acknowledge the coding assistance as required by your course policy.


In [ ]:
# Example only: uncomment if compact checkpoints were retained.
# restored = load_compact_checkpoint(ROOT / "damped/model.pt")
# print(restored.blocks.metadata())
# del restored
